# ECGMaxViT: A Hybrid CNN and MaxViT Model for Enhanced 12-Lead ECG Image Classification

**Architecture:** EfficientNetV2-S (CNN backbone) + MaxViT (Multi-Axis Vision Transformer) + MLP Fusion

**Dataset:** ECG Image Dataset (v1 + v2, excluding COVID-19) — 4 classes: MI, AHB, PMI, Normal

---

## 1. Install Dependencies

In [ ]:
# Install required libraries
!pip install timm --quiet
!pip install torchmetrics --quiet
!pip install seaborn --quiet
print("All dependencies installed successfully!")

## 2. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.transforms as transforms
from torchvision import models

import timm
from timm import create_model

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from torchmetrics import Accuracy, Precision, Recall, F1Score

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Dataset Setup & Exploration

**Dataset:** ECG Image Dataset from Mendeley Data
- Version 1 (DOI: 10.17632/gwbz3fsgp8.1): COVID-19, MI, AHB, PMI, Normal
- Version 2 (DOI: 10.17632/gwbz3fsgp8.2): MI, AHB, PMI, Normal
- We **exclude COVID-19** and combine both versions → 4 cardiac classes

In [ ]:
# ─── DATASET CONFIGURATION ───────────────────────────────────────────────────
# Adjust these paths according to your Kaggle dataset location
# The dataset should be added as a Kaggle dataset input

# Try common Kaggle paths for this dataset
POSSIBLE_ROOTS = [
    '/kaggle/input/ecg-image-data',
    '/kaggle/input/ecg-images-dataset',
    '/kaggle/input/electrocardiogram-ecg-image-dataset',
    '/kaggle/input',
]

DATA_ROOT = None
for path in POSSIBLE_ROOTS:
    if os.path.exists(path):
        DATA_ROOT = path
        print(f"Found dataset at: {DATA_ROOT}")
        break

if DATA_ROOT is None:
    print("Dataset not found at standard paths. Please update DATA_ROOT manually.")
    DATA_ROOT = '/kaggle/input'  # fallback

# Show dataset structure
print("\nDataset structure:")
for root, dirs, files in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = ' ' * 2 * (level + 1)
        img_count = len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))])
        if img_count > 0:
            print(f'{subindent}[{img_count} images]')

In [ ]:
# ─── AUTOMATIC DATASET DISCOVERY (UPDATED) ───────────────────────────────────
# Maps folder names (case-insensitive) to class labels
CLASS_MAPPING = {
    # 1. Check for History/PMI FIRST to prevent 'mi' keyword from stealing these
    'history of mi': 2, 'pmi': 2, 'history': 2, 'previous': 2,
    
    # 2. Other classes
    'myocardial infarction': 0, 'mi': 0, 
    'ahb': 1, 'arrhythmia': 1, 'abnormal': 1,
    'normal': 3,
}

CLASS_NAMES = ['MI', 'AHB', 'PMI', 'Normal']
NUM_CLASSES = 4
EXCLUDE_CLASSES = ['covid', 'covid-19', 'covid19']

def discover_dataset(root):
    """Auto-discover images and assign labels from folder structure."""
    image_paths = []
    labels = []
    skipped = []
    
    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    
    for dirpath, dirnames, filenames in os.walk(root):
        # CHANGE: Look at the FULL path in lowercase to catch nested folders
        full_path_lower = dirpath.lower()
        
        # Skip COVID folders
        if any(excl in full_path_lower for excl in EXCLUDE_CLASSES):
            if filenames: skipped.append(dirpath)
            continue
        
        # Check if any part of the path matches our mapping
        label = None
        for key, val in CLASS_MAPPING.items():
            if key in full_path_lower: # CHANGE: search in full path
                label = val
                break
        
        if label is None:
            continue
        
        for fname in filenames:
            if Path(fname).suffix.lower() in IMG_EXTS:
                image_paths.append(os.path.join(dirpath, fname))
                labels.append(label)
    
    print(f"Discovered {len(image_paths)} images across {NUM_CLASSES} classes")
    print(f"Skipped {len(skipped)} COVID folders")
    
    dist = Counter(labels)
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name}: {dist.get(i, 0)} images")
    
    return image_paths, labels

all_paths, all_labels = discover_dataset(DATA_ROOT)

## 4. Data Augmentation

Following the paper's methodology: color space transformations (BGRA, XYZ, HLS, RGBA) to balance class distribution.
**Note:** No rotation/flipping — positive and negative ECG deflections carry clinical meaning.

In [ ]:
def convert_color_space(img_bgr, mode):
    """
    Convert BGR image to different color spaces (for augmentation).
    Returns 3-channel BGR image for consistency.
    """
    if mode == 'BGRA':
        converted = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
        return cv2.cvtColor(converted, cv2.COLOR_BGRA2BGR)
    elif mode == 'XYZ':
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2XYZ)
    elif mode == 'HLS':
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HLS)
    elif mode == 'RGBA':
        converted = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGBA)
        return cv2.cvtColor(converted, cv2.COLOR_RGBA2BGR)
    else:
        return img_bgr


def augment_dataset(paths, labels, target_per_class=1500):
    """
    Augment dataset using color space transformations to balance classes.
    Augmentation strategies per class:
      MI  → BGRA + XYZ + HLS (3x)
      AHB → RGBA (1x)
      PMI → XYZ + HLS + RGBA (3x)
      Normal → XYZ (1x)
    """
    aug_strategies = {
        0: ['BGRA', 'XYZ', 'HLS'],    # MI
        1: ['RGBA'],                    # AHB
        2: ['XYZ', 'HLS', 'RGBA'],     # PMI
        3: ['XYZ'],                     # Normal
    }
    
    aug_paths  = list(paths)
    aug_labels = list(labels)
    aug_modes  = ['original'] * len(paths)  # track augmentation type
    
    label_to_paths = {}
    for p, l in zip(paths, labels):
        label_to_paths.setdefault(l, []).append(p)
    
    print("\nAugmentation summary:")
    for cls, modes in aug_strategies.items():
        cls_paths = label_to_paths.get(cls, [])
        for mode in modes:
            aug_paths.extend(cls_paths)
            aug_labels.extend([cls] * len(cls_paths))
            aug_modes.extend([mode] * len(cls_paths))
        total = len(cls_paths) * (1 + len(modes))
        print(f"  {CLASS_NAMES[cls]}: {len(cls_paths)} → {total} (+{len(modes)}x augmentations)")
    
    print(f"\nTotal dataset: {len(paths)} → {len(aug_paths)} images")
    return aug_paths, aug_labels, aug_modes


aug_paths, aug_labels, aug_modes = augment_dataset(all_paths, all_labels)

# Visualize class distribution before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

before_counts = Counter(all_labels)
after_counts  = Counter(aug_labels)

colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

axes[0].pie(
    [before_counts.get(i, 0) for i in range(4)],
    labels=CLASS_NAMES, autopct='%1.1f%%', colors=colors,
    startangle=90, textprops={'fontsize': 11}
)
axes[0].set_title('Before Augmentation (Class Imbalance)', fontsize=13, fontweight='bold')

axes[1].pie(
    [after_counts.get(i, 0) for i in range(4)],
    labels=CLASS_NAMES, autopct='%1.1f%%', colors=colors,
    startangle=90, textprops={'fontsize': 11}
)
axes[1].set_title('After Augmentation (Balanced Classes)', fontsize=13, fontweight='bold')

plt.suptitle('ECG Dataset Class Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Dataset Class & Data Splits

In [ ]:
# ════════════════════════════════════════════════════════
# SECTION 5 — FIXED TRANSFORMS + DATASET (replace old one)
# ════════════════════════════════════════════════════════

IMG_SIZE = 224

# FIX 1: Remove Grayscale from train (or keep it ONLY if you also add to val/test)
# ECG images are already near-grayscale — Grayscale was hurting because 
# val/test were still RGB, causing a train/val distribution mismatch.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    # ❌ REMOVED: transforms.Grayscale(num_output_channels=3)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class ECGDataset(Dataset):
    def __init__(self, paths, labels, modes, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.modes     = modes
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        label = self.labels[idx]
        mode  = self.modes[idx]

        # Load image
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            img_bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

        # Apply color space augmentation
        if mode != 'original':
            img_bgr = convert_color_space(img_bgr, mode)

        # Convert BGR → RGB → PIL for torchvision transforms
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)

        if self.transform:
            img_pil = self.transform(img_pil)

        return img_pil, torch.tensor(label, dtype=torch.long)


# ─── TRAIN / VAL / TEST SPLIT (60 / 20 / 20) ─────────────────────────────────
indices = list(range(len(aug_paths)))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.40, random_state=SEED,
    stratify=aug_labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED,
    stratify=[aug_labels[i] for i in temp_idx]
)

def subset(idx):
    return (
        [aug_paths[i] for i in idx],
        [aug_labels[i] for i in idx],
        [aug_modes[i] for i in idx]
    )

train_p, train_l, train_m = subset(train_idx)
val_p,   val_l,   val_m   = subset(val_idx)
test_p,  test_l,  test_m  = subset(test_idx)

train_dataset = ECGDataset(train_p, train_l, train_m, transform=train_transform)
val_dataset   = ECGDataset(val_p,   val_l,   val_m,   transform=val_test_transform)
test_dataset  = ECGDataset(test_p,  test_l,  test_m,  transform=val_test_transform)

BATCH_SIZE   = 8   # Physical batch (prevents OOM on T4/P100)
ACCUM_STEPS  = 4   # Gradient accumulation → effective batch = 8×4 = 32
NUM_WORKERS  = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Physical batch: {BATCH_SIZE} | Effective batch: {BATCH_SIZE * ACCUM_STEPS}")

# Dataset distribution table
dist_data = {
    'Class': CLASS_NAMES,
    'Train': [Counter(train_l).get(i,0) for i in range(4)],
    'Validation': [Counter(val_l).get(i,0) for i in range(4)],
    'Test': [Counter(test_l).get(i,0) for i in range(4)],
}
dist_data['Total'] = [a+b+c for a,b,c in zip(dist_data['Train'], dist_data['Validation'], dist_data['Test'])]
df_dist = pd.DataFrame(dist_data)
df_dist.loc[len(df_dist)] = ['Total'] + [df_dist[c].sum() for c in ['Train','Validation','Test','Total']]
print("\nDataset Distribution:")
print(df_dist.to_string(index=False))

## 6. ECGMaxViT Model Architecture

**Our novel hybrid model:**
- **ModCNN**: EfficientNetV2-S backbone (extracts 1280-dim local features via depthwise-separable convolutions)
- **MaxViT Module**: Multi-Axis Vision Transformer — combines local window attention + global grid attention (extracts 512-dim global features)
- **MLP Fusion**: Concatenates [1280 + 512] = 1792 features → FC(512) with GELU → FC(4) for classification

In [ ]:
# ─── ModCNN MODULE (EfficientNetV2-S backbone) ────────────────────────────────

class ModCNN(nn.Module):
    """
    Modified CNN module based on EfficientNetV2-S.
    Removes classification head; outputs 1280-dim feature vectors.
    Uses depthwise-separable convolutions for efficient local feature extraction.
    """
    def __init__(self, pretrained=True):
        super(ModCNN, self).__init__()
        # Load EfficientNetV2-S from timm
        backbone = create_model(
            'tf_efficientnetv2_s',
            pretrained=pretrained,
            num_classes=0,       # remove classifier head
            global_pool='avg'    # global average pooling
        )
        self.backbone = backbone
        self.feature_dim = backbone.num_features  # 1280

    def forward(self, x):
        # Returns (B, 1280) feature vector
        return self.backbone(x)


# ─── MaxViT MODULE ────────────────────────────────────────────────────────────

class MaxViTModule(nn.Module):
    """
    MaxViT (Multi-Axis Vision Transformer) module.
    Uses timm's maxvit_tiny_tf_224 — combines local window + global grid attention.
    Removes classification head; outputs 512-dim global feature vectors.
    """
    def __init__(self, pretrained=True):
        super(MaxViTModule, self).__init__()
        self.model = create_model(
            'maxvit_tiny_tf_224',
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )
        self.feature_dim = self.model.num_features  # 512

    def forward(self, x):
        # Returns (B, 512) global feature vector
        return self.model(x)


# ─── MLP FUSION MODULE ───────────────────────────────────────────────────────

class MLPFusion(nn.Module):
    """
    MLP module for early feature fusion and classification.
    Concatenates CNN features (1280) + MaxViT features (512) = 1792 dims.
    FC(1792 → 512) with GELU → Dropout → FC(512 → 4).
    """
    def __init__(self, cnn_dim=1280, vit_dim=512, hidden_dim=512, num_classes=4, dropout=0.3):
        super(MLPFusion, self).__init__()
        fused_dim = cnn_dim + vit_dim  # 1792

        self.fc1      = nn.Linear(fused_dim, hidden_dim)
        self.gelu     = nn.GELU()
        self.dropout  = nn.Dropout(dropout)
        self.fc2      = nn.Linear(hidden_dim, num_classes)
        self.norm     = nn.LayerNorm(fused_dim)

    def forward(self, f_cnn, f_vit):
        # Early fusion: concatenate features
        f_concat = torch.cat([f_cnn, f_vit], dim=1)       # (B, 1792)
        f_concat = self.norm(f_concat)                      # LayerNorm
        f_hidden = self.gelu(self.fc1(f_concat))            # GELU activation
        f_hidden = self.dropout(f_hidden)
        f_output = self.fc2(f_hidden)                       # logits (B, 4)
        return f_output


# ─── ECGMaxViT: FULL MODEL ───────────────────────────────────────────────────

class ECGMaxViT(nn.Module):
    """
    ECGMaxViT: Hybrid CNN + MaxViT model for 12-lead ECG image classification.

    Architecture:
        Input (224×224×3)
            │
            ├─► ModCNN (EfficientNetV2-S) ─► 1280-dim local features
            │
            ├─► MaxViT Module             ─► 512-dim global features
            │
            └─► MLP Fusion [1280+512=1792] ─► FC(512) ─► GELU ─► FC(4) ─► Softmax

    Output: 4-class probability (MI, AHB, PMI, Normal)
    """
    def __init__(self, num_classes=4, pretrained=True):
        super(ECGMaxViT, self).__init__()
        self.mod_cnn   = ModCNN(pretrained=pretrained)
        self.maxvit    = MaxViTModule(pretrained=pretrained)
        self.mlp_head  = MLPFusion(
            cnn_dim=self.mod_cnn.feature_dim,
            vit_dim=self.maxvit.feature_dim,
            hidden_dim=512,
            num_classes=num_classes
        )

    def forward(self, x):
        f_cnn = self.mod_cnn(x)    # Local features
        f_vit = self.maxvit(x)     # Global features
        out   = self.mlp_head(f_cnn, f_vit)
        return out

    def get_feature_dims(self):
        return {
            'CNN (EfficientNetV2-S)': self.mod_cnn.feature_dim,
            'MaxViT':                 self.maxvit.feature_dim,
            'Fused':                  self.mod_cnn.feature_dim + self.maxvit.feature_dim
        }


# ─── Instantiate model ───────────────────────────────────────────────────────
print("Building ECGMaxViT model...")
model = ECGMaxViT(num_classes=NUM_CLASSES, pretrained=True).to(device)

dims = model.get_feature_dims()
for name, dim in dims.items():
    print(f"  {name}: {dim} features")

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (approx):  {total_params * 4 / 1e6:.1f} MB")

## 7. Training Configuration

In [ ]:
# ════════════════════════════════════════════════════════
# SECTION 7 — FIXED HYPERPARAMETERS (replace old one)
# ════════════════════════════════════════════════════════

# FIX 2: Correct learning rate and weight decay
LEARNING_RATE = 2e-4      # ← Paper value (was 5e-5 — too low)
NUM_EPOCHS    = 100
WEIGHT_DECAY  = 1e-4      # ← 0.0001 (was 0.05 — 500x too high!)
WARMUP_EPOCHS = 5         # FIX 4: LR warmup prevents early instability
ACCUM_STEPS   = 4         # FIX 3: Gradient accumulation

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Warmup then cosine decay
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS   # linear warmup
    progress = (epoch - WARMUP_EPOCHS) / (NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("Fixed training configuration:")
print(f"  lr={LEARNING_RATE}  wd={WEIGHT_DECAY}  warmup={WARMUP_EPOCHS} epochs")
print(f"  Effective batch = {BATCH_SIZE} × {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}")


print("Training configuration:")
print(f"  Optimizer:     AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"  Scheduler:     CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"  Loss:          CrossEntropyLoss (label_smoothing=0.1)")
print(f"  Epochs:        {NUM_EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")

## 8. Training Loop

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# TRAINING SECTION — Complete (Fixed)
# Includes: train_epoch with gradient accumulation, eval_epoch, full training loop
# ════════════════════════════════════════════════════════════════════════════════

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# ── Hyperparameters ──────────────────────────────────────────────────────────
LEARNING_RATE  = 2e-4       # Paper value (was 5e-5 — too low)
NUM_EPOCHS     = 100
WEIGHT_DECAY   = 1e-4       # (was 0.05 — 500x too high)
WARMUP_EPOCHS  = 5          # Linear warmup before cosine decay
ACCUM_STEPS    = 4          # Gradient accumulation: 8 × 4 = effective batch 32
patience       = 20         # Early stopping patience (was 15 — too aggressive)

# ── Loss, Optimizer, Scheduler ───────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS          # linear warmup
    progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + np.cos(np.pi * progress))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("Training configuration:")
print(f"  Optimizer  : AdamW  lr={LEARNING_RATE}  wd={WEIGHT_DECAY}")
print(f"  Scheduler  : {WARMUP_EPOCHS}-epoch warmup → CosineDecay")
print(f"  Loss       : CrossEntropyLoss (label_smoothing=0.1)")
print(f"  Epochs     : {NUM_EPOCHS}  |  Early stop patience: {patience}")
print(f"  Phys batch : {BATCH_SIZE}  |  Accum steps: {ACCUM_STEPS}  |  Eff batch: {BATCH_SIZE * ACCUM_STEPS}")

# ── train_epoch — gradient accumulation ─────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, accum_steps):
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)

        # Scale loss so accumulated gradients equal a true mean over the
        # full effective batch (not just one physical mini-batch)
        loss = criterion(outputs, labels) / accum_steps
        loss.backward()

        # Track un-scaled loss for logging
        running_loss += loss.item() * accum_steps * imgs.size(0)

        preds    = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        # Step only when accumulation window is complete (or last batch)
        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

    epoch_loss = running_loss / total
    epoch_acc  = correct / total * 100
    return epoch_loss, epoch_acc


# ── eval_epoch — no gradient accumulation needed ─────────────────────────────
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    all_preds    = []
    all_labels   = []

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        loss    = criterion(outputs, labels)

        running_loss += loss.item() * imgs.size(0)

        preds    = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = correct / total * 100
    return epoch_loss, epoch_acc, all_preds, all_labels


# ── History tracking ─────────────────────────────────────────────────────────
history = {
    'train_loss'  : [],
    'train_acc'   : [],
    'val_loss'    : [],
    'val_acc'     : [],
    'val_precision': [],
    'val_recall'  : [],
    'val_f1'      : [],
}

best_val_acc     = 0.0
best_model_path  = 'ecg_maxvit_best.pth'
patience_counter = 0

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
      f"{'Val Loss':>8} | {'Val Acc':>7} | {'F1':>6} | {'LR':>9}")
print("─" * 78)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────────────
    tr_loss, tr_acc = train_epoch(
        model, train_loader, criterion, optimizer, ACCUM_STEPS
    )

    # ── Validate ─────────────────────────────────────────────────────────────
    vl_loss, vl_acc, vl_preds, vl_true = eval_epoch(
        model, val_loader, criterion
    )

    vl_prec = precision_score(vl_true, vl_preds, average='macro', zero_division=0) * 100
    vl_rec  = recall_score   (vl_true, vl_preds, average='macro', zero_division=0) * 100
    vl_f1   = f1_score       (vl_true, vl_preds, average='macro', zero_division=0) * 100

    # ── Scheduler step ───────────────────────────────────────────────────────
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── Record history ───────────────────────────────────────────────────────
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['val_precision'].append(vl_prec)
    history['val_recall'].append(vl_rec)
    history['val_f1'].append(vl_f1)

    # ── Checkpoint best model ─────────────────────────────────────────────────
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
        marker = "  ★ saved"
    else:
        patience_counter += 1
        marker = ""

    # ── Print every 5 epochs + epoch 1 ───────────────────────────────────────
    if epoch % 5 == 0 or epoch == 1:
        print(
            f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2f}% | "
            f"{vl_loss:>8.4f} | {vl_acc:>6.2f}% | {vl_f1:>5.2f}% | "
            f"{current_lr:>9.2e}{marker}"
        )

    # ── Early stopping ────────────────────────────────────────────────────────
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} "
              f"(no improvement for {patience} epochs)")
        break

print(f"\nBest Validation Accuracy : {best_val_acc:.2f}%")
print(f"Best model saved to      : {best_model_path}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE MODEL — Complete Package for Project Use
# Run this cell after training is complete
# Saves everything needed to load and use the model outside Kaggle
# ════════════════════════════════════════════════════════════════════════════════

import torch
import json
import os

# ── 1. Save full model checkpoint (weights + config) ─────────────────────────
# This saves everything needed to rebuild and reload the model
checkpoint = {
    # Model weights
    'model_state_dict'  : model.state_dict(),

    # Architecture config — needed to rebuild ECGMaxViT class
    'model_config': {
        'num_classes'   : NUM_CLASSES,
        'cnn_backbone'  : 'tf_efficientnetv2_s',
        'vit_backbone'  : 'maxvit_tiny_tf_224',
        'cnn_feat_dim'  : model.mod_cnn.feature_dim,   # 1280
        'vit_feat_dim'  : model.maxvit.feature_dim,    # 512
        'hidden_dim'    : 512,
        'dropout'       : 0.3,
        'img_size'      : 224,
    },

    # Class labels — needed for prediction output
    'class_names'       : CLASS_NAMES,   # ['MI', 'AHB', 'PMI', 'Normal']

    # Training info — useful for paper/report
    'training_info': {
        'epochs_trained'    : len(history['train_loss']),
        'best_val_acc'      : round(best_val_acc, 4),
        'test_acc'          : round(test_acc, 4),
        'test_precision'    : round(test_prec, 4),
        'test_recall'       : round(test_rec, 4),
        'test_f1'           : round(test_f1, 4),
        'optimizer'         : 'AdamW',
        'learning_rate'     : LEARNING_RATE,
        'weight_decay'      : WEIGHT_DECAY,
        'batch_size'        : BATCH_SIZE * ACCUM_STEPS,
        'dataset_size'      : len(aug_paths),
        'seed'              : SEED,
    },

    # Normalization stats — needed to preprocess images the same way
    'normalize_mean'    : [0.485, 0.456, 0.406],
    'normalize_std'     : [0.229, 0.224, 0.225],
}

SAVE_PATH = '/kaggle/working/ecg_maxvit_full.pth'
torch.save(checkpoint, SAVE_PATH)
print(f"✓ Full checkpoint saved  →  {SAVE_PATH}")
print(f"  File size: {os.path.getsize(SAVE_PATH)/1e6:.1f} MB")


# ── 2. Save FP16 (compressed) version ────────────────────────────────────────
import copy
model_fp16 = copy.deepcopy(model).half()
checkpoint_fp16 = checkpoint.copy()
checkpoint_fp16['model_state_dict'] = model_fp16.state_dict()
checkpoint_fp16['precision'] = 'float16'

SAVE_PATH_FP16 = '/kaggle/working/ecg_maxvit_fp16.pth'
torch.save(checkpoint_fp16, SAVE_PATH_FP16)
print(f"✓ FP16 checkpoint saved  →  {SAVE_PATH_FP16}")
print(f"  File size: {os.path.getsize(SAVE_PATH_FP16)/1e6:.1f} MB")
print(f"  Compression: {os.path.getsize(SAVE_PATH)/os.path.getsize(SAVE_PATH_FP16):.1f}x smaller")


# ── 3. Save class names as JSON (handy for web apps / APIs) ──────────────────
meta = {
    'class_names'    : CLASS_NAMES,
    'class_to_idx'   : {name: i for i, name in enumerate(CLASS_NAMES)},
    'idx_to_class'   : {i: name for i, name in enumerate(CLASS_NAMES)},
    'img_size'       : 224,
    'normalize_mean' : [0.485, 0.456, 0.406],
    'normalize_std'  : [0.229, 0.224, 0.225],
    'architecture'   : 'EfficientNetV2-S + MaxViT + MLP',
    'test_accuracy'  : round(test_acc, 2),
}
META_PATH = '/kaggle/working/ecg_maxvit_meta.json'
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"✓ Metadata JSON saved    →  {META_PATH}")


# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  FILES SAVED TO /kaggle/working/")
print("="*55)
print(f"  ecg_maxvit_full.pth   — full checkpoint (FP32)")
print(f"  ecg_maxvit_fp16.pth   — compressed checkpoint (FP16)")
print(f"  ecg_maxvit_meta.json  — class names + config")
print("="*55)
print("\nTo download: Kaggle sidebar → Output → Download")

## 9. Training History Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0,0].plot(epochs_range, history['train_loss'], label='Train Loss', color='#E74C3C', linewidth=2)
axes[0,0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#3498DB', linewidth=2)
axes[0,0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Accuracy
axes[0,1].plot(epochs_range, history['train_acc'], label='Train Acc', color='#E74C3C', linewidth=2)
axes[0,1].plot(epochs_range, history['val_acc'],   label='Val Acc',   color='#3498DB', linewidth=2)
axes[0,1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# F1, Precision, Recall
axes[1,0].plot(epochs_range, history['val_f1'],        label='F1-Score',  color='#2ECC71', linewidth=2)
axes[1,0].plot(epochs_range, history['val_precision'], label='Precision', color='#F39C12', linewidth=2)
axes[1,0].plot(epochs_range, history['val_recall'],    label='Recall',    color='#9B59B6', linewidth=2)
axes[1,0].set_title('Validation Metrics (F1 / Precision / Recall)', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Score (%)')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# Best metrics bar
best_ep = np.argmax(history['val_acc'])
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values  = [
    history['val_acc'][best_ep],
    history['val_precision'][best_ep],
    history['val_recall'][best_ep],
    history['val_f1'][best_ep]
]
bars = axes[1,1].bar(metrics, values, color=['#3498DB','#F39C12','#9B59B6','#2ECC71'], width=0.5)
for bar, val in zip(bars, values):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 2,
                   f'{val:.1f}%', ha='center', va='top', fontweight='bold', color='white', fontsize=11)
axes[1,1].set_title(f'Best Validation Metrics (Epoch {best_ep+1})', fontsize=13, fontweight='bold')
axes[1,1].set_ylabel('Score (%)')
axes[1,1].set_ylim(80, 101)
axes[1,1].grid(alpha=0.3, axis='y')

plt.suptitle('ECGMaxViT Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Final Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Evaluate on test set
_, test_acc, test_preds, test_true = eval_epoch(model, test_loader, criterion)

test_prec = precision_score(test_true, test_preds, average='macro', zero_division=0) * 100
test_rec  = recall_score(test_true,   test_preds, average='macro', zero_division=0) * 100
test_f1   = f1_score(test_true,       test_preds, average='macro', zero_division=0) * 100

# Evaluate on validation set (for paper comparison)
_, val_acc_final, val_preds_final, val_true_final = eval_epoch(model, val_loader, criterion)
val_prec_final = precision_score(val_true_final, val_preds_final, average='macro', zero_division=0) * 100
val_rec_final  = recall_score(val_true_final,    val_preds_final, average='macro', zero_division=0) * 100
val_f1_final   = f1_score(val_true_final,        val_preds_final, average='macro', zero_division=0) * 100

print("=" * 55)
print("         ECGMaxViT — FINAL RESULTS")
print("=" * 55)
print(f"{'Metric':<15} {'Validation':>12} {'Test':>12}")
print("-" * 42)
print(f"{'Accuracy':<15} {val_acc_final:>11.2f}% {test_acc:>11.2f}%")
print(f"{'Precision':<15} {val_prec_final:>11.2f}% {test_prec:>11.2f}%")
print(f"{'Recall':<15} {val_rec_final:>11.2f}% {test_rec:>11.2f}%")
print(f"{'F1-Score':<15} {val_f1_final:>11.2f}% {test_f1:>11.2f}%")
print("=" * 55)

# Per-class report
print("\nDetailed Classification Report (Test Set):")
print(classification_report(test_true, test_preds, target_names=CLASS_NAMES))

## 11. Confusion Matrices

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5, linecolor='white',
        annot_kws={'size': 13, 'weight': 'bold'}
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Labels', fontsize=11)
    ax.set_ylabel('True Labels', fontsize=11)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_confusion_matrix(val_true_final, val_preds_final,
                      'Validation Set Confusion Matrix', axes[0])
plot_confusion_matrix(test_true, test_preds,
                      'Test Set Confusion Matrix', axes[1])

plt.suptitle('ECGMaxViT — Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Per-Class Performance Bar Chart

In [ ]:
# Per-class metrics
prec_per = precision_score(test_true, test_preds, average=None, zero_division=0) * 100
rec_per  = recall_score(test_true,   test_preds, average=None, zero_division=0) * 100
f1_per   = f1_score(test_true,       test_preds, average=None, zero_division=0) * 100

x = np.arange(len(CLASS_NAMES))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Test set per-class
bars1 = ax1.bar(x - width, prec_per, width, label='Precision', color='#F39C12', alpha=0.85)
bars2 = ax1.bar(x,         rec_per,  width, label='Recall',    color='#9B59B6', alpha=0.85)
bars3 = ax1.bar(x + width, f1_per,   width, label='F1-Score',  color='#2ECC71', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax1.set_xticks(x); ax1.set_xticklabels(CLASS_NAMES, fontsize=11)
ax1.set_ylim(80, 103); ax1.set_ylabel('Score (%)')
ax1.set_title('Per-Class Metrics (Test Set)', fontsize=13, fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3, axis='y')

# Accuracy comparison — test vs val
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
val_scores  = [val_acc_final,  val_prec_final,  val_rec_final,  val_f1_final]
test_scores = [test_acc,       test_prec,       test_rec,       test_f1]

x2 = np.arange(len(metrics_names))
ax2.bar(x2 - 0.2, val_scores,  0.38, label='Validation', color='#3498DB', alpha=0.85)
ax2.bar(x2 + 0.2, test_scores, 0.38, label='Test',       color='#E74C3C', alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(metrics_names, fontsize=11)
ax2.set_ylim(80, 103); ax2.set_ylabel('Score (%)')
ax2.set_title('Val vs Test Performance', fontsize=13, fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3, axis='y')

for bars, scores in [(ax2.containers[0], val_scores), (ax2.containers[1], test_scores)]:
    for bar, val in zip(bars, scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('ECGMaxViT — Performance Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Comparison with Other Models (ML & DL Baselines)

In [ ]:
# Ensure all constants are defined for the baselines
NUM_CLASSES = 4 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Double check that your DataLoaders are still in memory
# (They must be named train_loader, val_loader, and test_loader)
# ════════════════════════════════════════════════════════════════════════════════
# BASELINE COMPARISON — Complete Rewrite
# Models: SimpleCNN, DeepCNN, SimpleViT, SimpleHybrid (CNN+ViT), ECGMaxViT (ours)
# No external model zoo needed — all built from scratch with PyTorch only
# ════════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# ── 1. Simple 2D CNN ─────────────────────────────────────────────────────────
# Basic CNN with 3 conv blocks — the most standard baseline
class SimpleCNN(nn.Module):
    """
    Basic 3-block 2D CNN.
    Conv → BN → ReLU → MaxPool, repeated 3 times, then FC classifier.
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # 224 → 112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # 112 → 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                    # 56 → 28
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),            # 28×28 → 1×1
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── 2. Deep CNN (VGG-style) ───────────────────────────────────────────────────
# Deeper CNN with 5 blocks — closer to VGG architecture
class DeepCNN(nn.Module):
    """
    5-block VGG-style deep CNN.
    Each block: Conv → BN → ReLU → Conv → BN → ReLU → MaxPool.
    """
    def __init__(self, num_classes=4):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3,   32),    # 224 → 112
            conv_block(32,  64),    # 112 → 56
            conv_block(64,  128),   # 56  → 28
            conv_block(128, 256),   # 28  → 14
            conv_block(256, 512),   # 14  → 7
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── 3. Simple Vision Transformer (ViT) ───────────────────────────────────────
# Lightweight ViT built from scratch — patch embedding + transformer encoder
class PatchEmbedding(nn.Module):
    """Split image into patches and project to embedding dimension."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # Conv2d with stride=patch_size acts as patch tokenizer
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)            # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)            # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)      # (B, num_patches, embed_dim)
        return x


class SimpleViT(nn.Module):
    """
    Lightweight Vision Transformer (ViT).
    Patch size 16×16, embed_dim=192, 4 transformer blocks, 4 attention heads.
    CLS token for classification.
    """
    def __init__(self, img_size=224, patch_size=16, embed_dim=192,
                 num_heads=4, num_layers=4, mlp_ratio=4, num_classes=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        num_patches = (img_size // patch_size) ** 2

        # Learnable CLS token and positional encoding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # Transformer encoder blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model    = embed_dim,
            nhead      = num_heads,
            dim_feedforward = embed_dim * mlp_ratio,
            dropout    = dropout,
            activation = 'gelu',
            batch_first= True,
            norm_first = True,      # Pre-LN (more stable)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)                                 # (B, N, D)

        cls = self.cls_token.expand(B, -1, -1)                  # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)                        # (B, N+1, D)
        x   = self.pos_drop(x + self.pos_embed)

        x   = self.transformer(x)                               # (B, N+1, D)
        x   = self.norm(x[:, 0])                                # CLS token only
        return self.head(x)


# ── 4. Simple CNN + ViT Hybrid ────────────────────────────────────────────────
# CNN extracts local features → flattened → treated as tokens by Transformer
class SimpleCNNViT(nn.Module):
    """
    Simple hybrid CNN + Transformer.
    CNN backbone extracts spatial feature maps → reshaped as token sequence
    → fed into a small Transformer encoder → FC classifier.
    Bridges local CNN features with global self-attention.
    """
    def __init__(self, num_classes=4, embed_dim=256, num_heads=4,
                 num_layers=2, dropout=0.1):
        super().__init__()

        # CNN backbone — outputs (B, 256, 14, 14)
        self.cnn = nn.Sequential(
            nn.Conv2d(3,   32,  kernel_size=3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(True), nn.MaxPool2d(2),   # 112
            nn.Conv2d(32,  64,  kernel_size=3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(True), nn.MaxPool2d(2),   # 56
            nn.Conv2d(64,  128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),   # 28
            nn.Conv2d(128, embed_dim, kernel_size=3, padding=1), nn.BatchNorm2d(embed_dim), nn.ReLU(True), nn.MaxPool2d(2),  # 14
        )

        # Positional embedding for 14×14 = 196 tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, 196 + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = num_heads,
            dim_feedforward = embed_dim * 2,
            dropout         = dropout,
            activation      = 'gelu',
            batch_first     = True,
            norm_first      = True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

        # Classifier
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B  = x.size(0)
        x  = self.cnn(x)                            # (B, 256, 14, 14)
        x  = x.flatten(2).transpose(1, 2)           # (B, 196, 256) — tokens

        cls = self.cls_token.expand(B, -1, -1)      # (B, 1, 256)
        x   = torch.cat([cls, x], dim=1)            # (B, 197, 256)
        x   = x + self.pos_embed

        x   = self.transformer(x)
        x   = self.norm(x[:, 0])                    # CLS token
        return self.head(x)


# ════════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTION FOR BASELINES
# ════════════════════════════════════════════════════════════════════════════════

def train_baseline(model_name, model_obj, num_epochs=30):
    """
    Train a baseline model for num_epochs and return test set metrics.
    Uses same train/val/test loaders as ECGMaxViT for fair comparison.
    """
    print(f"\n  Training {model_name}...")

    m    = model_obj.to(device)
    opt  = AdamW(m.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=num_epochs, eta_min=1e-6)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_acc   = 0.0
    best_state = None

    for ep in range(1, num_epochs + 1):
        # ── train ──────────────────────────────────────────────────────────
        m.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out  = m(imgs)
            loss = crit(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            opt.step()
        sched.step()

        # ── validate ───────────────────────────────────────────────────────
        _, vacc, _, _ = eval_epoch(m, val_loader, crit)
        if vacc > best_acc:
            best_acc   = vacc
            best_state = {k: v.clone() for k, v in m.state_dict().items()}

        if ep % 10 == 0:
            print(f"    Epoch {ep:>3}/{num_epochs}  val_acc={vacc:.1f}%  best={best_acc:.1f}%")

    # ── test with best checkpoint ───────────────────────────────────────────
    m.load_state_dict(best_state)
    _, tacc, tpreds, ttrue = eval_epoch(m, test_loader, crit)

    tprec = precision_score(ttrue, tpreds, average='macro', zero_division=0) * 100
    trec  = recall_score   (ttrue, tpreds, average='macro', zero_division=0) * 100
    tf1   = f1_score       (ttrue, tpreds, average='macro', zero_division=0) * 100

    # Count parameters
    n_params = sum(p.numel() for p in m.parameters()) / 1e6

    print(f"  ✓ {model_name:<22}  Acc={tacc:.1f}%  Pre={tprec:.1f}%  "
          f"Rec={trec:.1f}%  F1={tf1:.1f}%  Params={n_params:.1f}M")

    return {
        'Model'      : model_name,
        'Accuracy%'  : round(tacc,  1),
        'Precision%' : round(tprec, 1),
        'Recall%'    : round(trec,  1),
        'F1-Score%'  : round(tf1,   1),
        'Params (M)' : round(n_params, 1),
    }


# ════════════════════════════════════════════════════════════════════════════════
# RUN ALL BASELINES
# ════════════════════════════════════════════════════════════════════════════════

BASELINE_EPOCHS = 30   # 30 epochs each — fair comparison

baseline_models = {
    'Simple CNN (3-block)'    : SimpleCNN(num_classes=NUM_CLASSES),
    'Deep CNN (VGG-style)'    : DeepCNN(num_classes=NUM_CLASSES),
    'Simple ViT'              : SimpleViT(num_classes=NUM_CLASSES),
    'Simple CNN+ViT Hybrid'   : SimpleCNNViT(num_classes=NUM_CLASSES),
}

print("=" * 65)
print("  BASELINE TRAINING — each model trained for 30 epochs")
print("=" * 65)

results = []
for name, model_obj in baseline_models.items():
    try:
        result = train_baseline(name, model_obj, num_epochs=BASELINE_EPOCHS)
        results.append(result)
    except Exception as e:
        print(f"  ✗ {name} failed: {e}")


# ── Add ECGMaxViT result (already evaluated above) ──────────────────────────
# Run evaluation now if test_acc not yet defined
model.load_state_dict(torch.load(best_model_path, map_location=device))
_, test_acc, test_preds, test_true = eval_epoch(model, test_loader, criterion)
test_prec = precision_score(test_true, test_preds, average='macro', zero_division=0) * 100
test_rec  = recall_score   (test_true, test_preds, average='macro', zero_division=0) * 100
test_f1   = f1_score       (test_true, test_preds, average='macro', zero_division=0) * 100
our_params = sum(p.numel() for p in model.parameters()) / 1e6

results.append({
    'Model'      : 'ECGMaxViT (Ours)',
    'Accuracy%'  : round(test_acc,  1),
    'Precision%' : round(test_prec, 1),
    'Recall%'    : round(test_rec,  1),
    'F1-Score%'  : round(test_f1,   1),
    'Params (M)' : round(our_params, 1),
})


# ════════════════════════════════════════════════════════════════════════════════
# PRINT COMPARISON TABLE
# ════════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(results)

print("\n")
print("=" * 75)
print("                   COMPARISON WITH BASELINE MODELS")
print("=" * 75)
print(df.to_string(index=False))
print("=" * 75)

# Highlight best in each column
print("\nBest per metric:")
for col in ['Accuracy%', 'Precision%', 'Recall%', 'F1-Score%']:
    best_idx = df[col].idxmax()
    print(f"  {col:<14} → {df.loc[best_idx, 'Model']} ({df.loc[best_idx, col]}%)")


# ════════════════════════════════════════════════════════════════════════════════
# PLOT COMPARISON
# ════════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models  = df['Model'].tolist()
x       = np.arange(len(models))
metrics = ['Accuracy%', 'Precision%', 'Recall%', 'F1-Score%']
colors  = ['#3498DB', '#F39C12', '#9B59B6', '#2ECC71']
width   = 0.2

# Bar chart — all 4 metrics side by side
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = axes[0].bar(x + (i - 1.5) * width,
                       df[metric], width,
                       label=metric.replace('%',''), color=color, alpha=0.85)

# Highlight our model
axes[0].axvspan(len(models) - 1.5, len(models) - 0.5,
                alpha=0.08, color='gold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(40, 105)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('All Models — All Metrics', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, axis='y')

# Accuracy-only line chart to show progression
axes[1].plot(models, df['Accuracy%'], marker='o', linewidth=2.5,
             color='#E74C3C', markersize=8, zorder=3)
for i, (m, v) in enumerate(zip(models, df['Accuracy%'])):
    axes[1].annotate(f'{v}%', (i, v), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(range(len(models)))
axes[1].set_xticklabels(models, rotation=20, ha='right', fontsize=9)
axes[1].set_ylim(40, 105)
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Accuracy Progression by Model Complexity', fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].fill_between(range(len(models)), df['Accuracy%'], alpha=0.08, color='#E74C3C')

plt.suptitle('ECGMaxViT vs Baseline Models — Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()